In [ ]:
from openai import OpenAI 
import numpy as np  

In [ ]:
def create_embeddings(text, model="text-embedding-ada-002"):
    """
    使用指定的OpenAI模型为给定文本创建嵌入。

    参数:
    text (str): 要为其创建嵌入的输入文本。
    model (str): 用于创建嵌入的模型。默认值为"text-embedding-ada-002"。

    返回:
    dict: 包含嵌入的OpenAI API回复。
    """
    # 使用指定模型为输入文本创建嵌入
    response = client.embeddings.create(
        model=model,
        input=text
    )

    return response  # 返回包含嵌入的回复

# 为文本块创建嵌入
response = create_embeddings(text_chunks)

In [ ]:
def cosine_similarity(vec1, vec2):
    """
    计算两个向量之间的余弦相似度。

    参数:
    vec1 (np.ndarray): 第一个向量。
    vec2 (np.ndarray): 第二个向量。

    返回:
    float: 两个向量之间的余弦相似度。
    """
    # 计算两个向量的点积并除以其范数的乘积
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

In [ ]:
def context_enriched_search(query, text_chunks, embeddings, k=1, context_size=1):
    """
    获取最相关的片段及其邻近片段。

    参数:
    query (str): 搜索查询。
    text_chunks (List[str]): 文本片段列表。
    embeddings (List[dict]): 片段嵌入列表。
    k (int): 要检索的相关片段数量。
    context_size (int): 包含的邻近片段数量。

    返回:
    List[str]: 带有上下文信息的相关文本片段。
    """
    # 将查询转换为嵌入向量
    query_embedding = create_embeddings(query).data[0].embedding
    similarity_scores = []

    # 计算查询与每个文本片段嵌入之间的相似度分数
    for i, chunk_embedding in enumerate(embeddings):
        # 计算查询嵌入与当前片段嵌入之间的余弦相似度
        similarity_score = cosine_similarity(np.array(query_embedding), np.array(chunk_embedding.embedding))
        # 将索引和相似度分数存储为元组
        similarity_scores.append((i, similarity_score))

    # 按相似度分数降序排序（最高相似度优先）
    similarity_scores.sort(key=lambda x: x[1], reverse=True)

    # 获取最相关片段的索引
    top_index = similarity_scores[0][0]

    # 定义上下文包含的范围
    # 确保不超出文本片段的起始或结束边界
    start = max(0, top_index - context_size)
    end = min(len(text_chunks), top_index + context_size + 1)

    # 返回相关片段及其邻近上下文片段
    return [text_chunks[i] for i in range(start, end)]